# Fast Fraud Detection with LightGBM

In this notebook, we'll implement a quick fraud detection model using LightGBM, which is much faster than RandomForest and typically performs better. We'll use:

1. LightGBM - Fast gradient boosting
2. Feature importance for quick insights
3. Cross-validation for reliable results
4. Early stopping for faster training

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# For reproducibility
np.random.seed(42)

In [ ]:
# Load and prepare data
df = pd.read_csv('c:/Users/anonymous/Downloads/fraud_oracle.csv')
print(f"Dataset shape: {df.shape}")
print("\nSample data:")
display(df.head())

# Check fraud distribution
fraud_dist = df['FraudFound_P'].value_counts(normalize=True)
print("\nFraud Distribution:")
print(fraud_dist)

In [ ]:
# Enhanced data preprocessing
def preprocess_data(df):
    df = df.copy()
    
    # Create label encoders
    le_dict = {}
    categorical_cols = df.select_dtypes(include=['object']).columns
    
    # Encode categorical variables
    for col in categorical_cols:
        if col not in ['PolicyNumber', 'RepNumber']:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            le_dict[col] = le
    
    # Feature engineering
    # Age-based features
    if 'VehicleAge' in df.columns:
        df['VehicleAgeSquared'] = df['VehicleAge'] ** 2
    
    # Amount-based features
    if 'ClaimAmount' in df.columns and 'VehiclePrice' in df.columns:
        df['ClaimAmountRatio'] = df['ClaimAmount'] / df['VehiclePrice']
    
    # Time-based features
    if 'PolicyDuration' in df.columns:
        df['PolicyDurationMonths'] = df['PolicyDuration'] / 30  # Assuming days
        df['IsNewPolicy'] = (df['PolicyDuration'] < 90).astype(int)
    
    # Interaction features
    if 'VehicleAge' in df.columns and 'ClaimAmount' in df.columns:
        df['AgeAmount_Interaction'] = df['VehicleAge'] * df['ClaimAmount']
    
    # Save encoders
    import joblib
    joblib.dump(le_dict, 'models/fraud_detector/label_encoders.joblib')
    
    return df

# Preprocess data with new features
df_processed = preprocess_data(df.copy())

# Split features and target
X = df_processed.drop(['FraudFound_P', 'PolicyNumber', 'RepNumber'], axis=1)
y = df_processed['FraudFound_P']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)
print("\nNew features added:", [col for col in X.columns if col not in df.columns])

In [ ]:
# Improved model training with cross-validation and hyperparameter tuning
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import roc_auc_score
import numpy as np

# Define model with better parameters
model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=7,
    num_leaves=31,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
    boosting_type='goss',  # Gradient-based One-Side Sampling for faster training
    reg_alpha=0.1,  # L1 regularization
    reg_lambda=0.1,  # L2 regularization
)

# Perform k-fold cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

print("Cross-validation ROC-AUC scores:", cv_scores)
print(f"Mean ROC-AUC: {cv_scores.mean():.3f} (+/- {cv_scores.std() * 2:.3f})")

# Train final model with early stopping
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric=['auc', 'binary_logloss'],
    early_stopping_rounds=50,
    verbose=50
)

# Make predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Find optimal threshold using ROC curve
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

# Apply optimal threshold
y_pred_optimal = (y_pred_proba >= optimal_threshold).astype(int)

# Print results
print("\nResults with default threshold (0.5):")
print(classification_report(y_test, y_pred))

print("\nResults with optimal threshold ({:.3f}):".format(optimal_threshold))
print(classification_report(y_test, y_pred_optimal))

print("\nROC-AUC Score:", roc_auc_score(y_test, y_pred_proba))

# Save the model and threshold
import joblib
joblib.dump(model, 'models/fraud_detector/model.joblib')
joblib.dump(optimal_threshold, 'models/fraud_detector/optimal_threshold.joblib')

In [ ]:
# Plot feature importance
feature_imp = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_imp.head(20))
plt.title('Top 20 Most Important Features')
plt.tight_layout()
plt.show()

# Print top features
print("\nTop 10 Most Important Features:")
print(feature_imp.head(10))

In [ ]:
# Plot ROC curve
from sklearn.metrics import roc_curve, auc
plt.figure(figsize=(8, 6))
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, color='darkorange', lw=2, 
         label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

# Plot confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred_optimal)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix (with optimal threshold)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()